# Embedding Space & Vector DB Landscape (Local & In-Memory Edition)

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Today is the tooling-week breather. Day 1 is **conceptual**: we look at what different open-weight embedding models produce locally, build intuition for semantic geometry, tour the vector-DB landscape, and end by executing local in-memory Qdrant vector retrieval. Day 2 goes deeper on indices, similarity metrics, and does a full migration walkthrough.

This is a **standalone demo** — nothing to do with the capstone. That's on purpose: today we learn embeddings and Qdrant in isolation. Migrating your capstone is Track B (take-home).

> **Vocareum Execution Note:**
> Previously, cloud-based models (`text-embedding-3-small`/`large`) were used via third-party API keys. In this revised workflow, we run **100% open-weight local models** using `sentence-transformers` and an in-memory Qdrant vector engine. This eliminates API key requirements, network dependencies, and per-token cloud costs.
> 
> 

---

### Key Specifications

* **Corpus:** 10 Enterprise IT & DevOps incident tickets across 5 categories. Diverse enough that semantic clustering is interesting to explore.


* **Local Embedding Models:**
1. `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, fast & lightweight)


2. `nomic-ai/nomic-embed-text-v1.5` (768 dimensions, asymmetric retrieval with `search_document: ` / `search_query: ` prefixes)




* **Vector Store:** Local In-Memory Qdrant (`QdrantClient(":memory:")`)


* **Cost per full run:** **$0.00** (Zero external API dependencies; runs fully on local CPU/GPU).



---

### Updated Notebook Flow

* **Cells 1–2:** Setup (transformers compatibility patch & hardware check) + Tickets corpus


* **Cell 3:** Quick recap — local embedding generation (`all-MiniLM-L6-v2`)


* **Cells 4–6:** Compare `all-MiniLM-L6-v2` (384d) vs `nomic-embed-text-v1.5` (768d), model discrimination benchmark, & storage math


* **Cells 7–9:** Semantic geometry — local batch vector generation, $10 \times 10$ pairwise cosine similarity matrix, & nearest-neighbor ranking


* **Cell 10:** Vector DB landscape (FAISS, Chroma, Qdrant, pgvector reference cards)


* **Cells 11–14:** Local In-Memory Qdrant — connect (`:memory:`), create collection (384d), upsert local points, & execute top-K semantic search


* **Cell 15:** Wrap & architectural summary

## Cell 1 — Setup

Imports local embedding libraries (`sentence_transformers`) alongside Qdrant credentials. 
Replaces external OpenAI API dependencies with fully local, open-weight model architectures.

In [1]:
#Pre-install: brew install qd
%pip install torch
%pip install sentence_transformers
%pip install sentence-transformers torch numpy
#%pip install QdrantClient
%pip install load_dotenv
%pip install qdrant-client python-dotenv sentence-transformers torch einops
import os
import time
from typing import List, Dict, Any, Tuple
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Hardware compute device auto-detection
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# Model identifier constants
MODEL_MINILM = "sentence-transformers/all-MiniLM-L6-v2"  # 384 dims, fast & lightweight
MODEL_NOMIC  = "nomic-ai/nomic-embed-text-v1.5"         # 768 dims, high accuracy with prefix support

print("Local setup complete.")
print(f"Execution device: {device.upper()}")
print("Environment: 100% Offline / Local Execution")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Local setup complete.
Execution device: MPS
Environment: 100% Offline / Local Execution


-----


### Cell 1 Explanation — Setup & Environment Configuration

1) **What was performed in the cell above?**
Applies an in-memory monkey-patch for `nomic-ai/nomic-embed-text-v1.5` to ensure compatibility with `transformers >= 4.41.0`. It auto-detects hardware acceleration (CUDA, MPS, or CPU) and defines string constants for local open-weight embedding models (`all-MiniLM-L6-v2` and `nomic-embed-text-v1.5`).

2) **Discussion Moment**
> **Why patch `transformers` dynamically in memory?**
> Pinning older versions (`transformers<4.41.0`) often causes package dependency conflicts in modern environments like Python 3.13. Monkey-patching `get_extended_attention_mask` at runtime lets Nomic v1.5 run without downgrading system packages.

3) **Key points to consider**
* **Zero API Keys:** No `OPENAI_API_KEY` or third-party cloud credentials are required.
* **Hardware Acceleration:** Auto-routing tensors to Apple Silicon (`MPS`) or GPU dramatically speeds up local matrix computations compared to CPU fallback.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Loaded `.env`, asserted `OPENAI_API_KEY`, `QDRANT_URL`, and `QDRANT_API_KEY`, initialized `client = OpenAI()`, and assigned `EMBED_SMALL` (1536d) and `EMBED_LARGE` (3072d).
* **Migrated Stack:** Replaced commercial APIs with local `SentenceTransformer` imports and hardware device detection (`MPS`).

5) **What is shared in Execution Logs when Run the Cell?**
* **Package Inspections:** Shows `%pip install` output confirming `torch`, `sentence_transformers`, `qdrant-client`, `python-dotenv`, and `einops` are satisfied.
* **Hardware Detection:** Prints `Execution device: MPS`, confirming hardware acceleration on Apple Silicon.
* **Execution Mode:** Prints `Environment: 100% Offline / Local Execution`.

## Cell 2 — The corpus

10 IT tickets across 5 categories (Auth, Database, Network, Billing, CI/CD).
Used as a test corpus to evaluate local semantic clustering and vector space alignment.

In [2]:
documents = [
    # Auth & Access
    {"id": "tkt_auth_01", "category": "auth", "text": "User unable to authenticate via Okta SSO; receiving HTTP 401 unauthorized error on dashboard login."},
    {"id": "tkt_auth_02", "category": "auth", "text": "Active Directory group permissions dropped for new hires; missing IAM access tokens for AWS production."},
    # Database & Storage
    {"id": "tkt_db_01", "category": "database", "text": "PostgreSQL primary instance experiencing connection pool exhaustion during peak read operations."},
    {"id": "tkt_db_02", "category": "database", "text": "Redis cache cluster high memory utilization causing latency spikes in query resolution."},
    # Network & Security
    {"id": "tkt_net_01", "category": "network", "text": "VPN gateway connection timeout reported for remote workforce accessing internal subnet."},
    {"id": "tkt_net_02", "category": "network", "text": "Ingress controller dropping TLS handshake traffic following certificate auto-renewal fail."},
    # Billing & Licensing
    {"id": "tkt_bill_01", "category": "billing", "text": "Monthly SaaS license seat count overage alert triggered for Enterprise Salesforce tier."},
    {"id": "tkt_bill_02", "category": "billing", "text": "Invoice disparity on AWS cloud expenditure breakdown due to unreserved EC2 instances."},
    # CI/CD & Deployment
    {"id": "tkt_cicd_01", "category": "cicd", "text": "Jenkins pipeline step failing during container image build due to missing base Docker layer."},
    {"id": "tkt_cicd_02", "category": "cicd", "text": "Kubernetes deployment rollout stuck in CrashLoopBackOff after release v2.4 deployment."}
]

print(f"Loaded {len(documents)} IT tickets across 5 categories.\n")
for d in documents:
    print(f"  {d['id']:11s}  ({d['category']:8s})  {len(d['text']):3d} chars")

Loaded 10 IT tickets across 5 categories.

  tkt_auth_01  (auth    )   99 chars
  tkt_auth_02  (auth    )  103 chars
  tkt_db_01    (database)   96 chars
  tkt_db_02    (database)   87 chars
  tkt_net_01   (network )   87 chars
  tkt_net_02   (network )   90 chars
  tkt_bill_01  (billing )   87 chars
  tkt_bill_02  (billing )   85 chars
  tkt_cicd_01  (cicd    )   92 chars
  tkt_cicd_02  (cicd    )   86 chars


-----


### Cell 2 Explanation — The Ticket Corpus

1) **What was performed in the cell above?**
Instantiates a 10-document text corpus distributed evenly across 5 IT infrastructure domains: Auth, Database, Network, Billing, and CI/CD.

2) **Discussion Moment**
> **Why is a 10-document corpus ideal for semantic analysis?**
> Ten items are enough to test cross-category clustering while remaining small enough to print and inspect complete $10 \times 10$ similarity matrices and nearest-neighbor rankings without clutter.

3) **Key points to consider**
* **Balanced Lengths:** Passages are kept within ~160–225 characters to prevent text length variations from skewing vector norms.
* **Metadata Structure:** Each item contains structured key-value pairs (`id`, `category`, `text`) ready for vector DB payload indexing.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Used the exact same 10-document dataset.
* **Migrated Stack:** Maintained the exact same corpus structure to provide a 1:1 baseline comparison against local models.

5) **What is shared in Execution Logs when Run the Cell?**
* **Corpus Summary:** Prints `Loaded 10 IT tickets across 5 categories.`
* **Character Counts:** Lists each ticket ID, category, and text length.

## Cell 3 — Quick recap: local embedding generation

Loads local models into memory and transforms input text into a fixed-length vector of floating point numbers.

In [3]:
# Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

sample_sentence = "User unable to authenticate via Okta SSO."

def embed_one_minilm(text: str) -> list[float]:
    embedding = minilm_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

vec = embed_one_minilm(sample_sentence)

print(f"Sentence: {sample_sentence!r}")
print(f"Model:    {MODEL_MINILM}")
print(f"Vector length (dim): {len(vec)}")
print(f"First 8 dims: {[round(x, 6) for x in vec[:8]]}")
print(f"Type of each element: {type(vec[0]).__name__}")
print("\nTakeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence: 'User unable to authenticate via Okta SSO.'
Model:    sentence-transformers/all-MiniLM-L6-v2
Vector length (dim): 384
First 8 dims: [0.006153, 0.071634, -0.020263, -0.009423, -0.096628, -0.029596, 0.050656, -0.029087]
Type of each element: float

Takeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.


----


### Cell 3 Explanation — Quick Recap: Local Embedding Generation

1) **What was performed in the cell above?**
Loads `all-MiniLM-L6-v2` locally and transforms a single sample sentence into a 384-dimensional $L_2$-normalized floating-point vector.

2) **Discussion Moment**
> **Why is $L_2$ normalization (`normalize_embeddings=True`) essential?**
> When vectors are normalized to unit length ($\Vert{}v\Vert{}_2 = 1.0$), calculating cosine similarity simplifies mathematically to a basic dot product ($\vec{a} \cdot \vec{b}$), eliminating expensive square-root operations during search.

3) **Key points to consider**
* **Vector Unreadability:** Floating-point numbers have no human-interpretable meaning in isolation; semantic context exists entirely in spatial relationships relative to other vectors.
* **Lightweight Footprint:** A 384-dimensional vector takes up only 1.5 KB of memory ($384 \times 4$ bytes).

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Called `client.embeddings.create(model="text-embedding-3-small", input=[text])` over network HTTP, returning a 1536-dimensional vector.
* **Migrated Stack:** Runs `minilm_model.encode()` locally in RAM, returning a 384-dimensional vector in milliseconds.

5) **What is shared in Execution Logs when Run the Cell?**
* **HF Hub Warning:** Notes an unauthenticated request to Hugging Face Hub while downloading model weights.
* **Vector Specifications:** Confirms model `sentence-transformers/all-MiniLM-L6-v2`, `Vector length (dim): 384`, element type `float`, and prints the first 8 float values.

## Cell 4 — Compare: MiniLM (384d) vs Nomic v1.5 (768d)

We load both open-weight models locally to compare dimensions, initialization parameters, and latency.

| Model | Dimensions | Target Use Case | Prefix Required? |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Ultra-fast local execution, low memory | No |
| `nomic-embed-text-v1.5` | 768 | High-accuracy retrieval, long context | Yes (`search_document: ` / `search_query: `) |

In [4]:
import time
import torch
import transformers
from sentence_transformers import SentenceTransformer

# --- SELF-CONTAINED MONKEY PATCH FOR NOMIC V1.5 ---
def _patch_get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        extended_attention_mask = attention_mask

    target_dtype = dtype if dtype is not None else getattr(self, "dtype", torch.float32)
    extended_attention_mask = extended_attention_mask.to(dtype=target_dtype)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(target_dtype).min
    return extended_attention_mask

if not hasattr(transformers.PreTrainedModel, "get_extended_attention_mask"):
    transformers.PreTrainedModel.get_extended_attention_mask = _patch_get_extended_attention_mask
# ---------------------------------------------------

# 1. Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

# 2. Instantiate Nomic v1.5 locally
nomic_model = SentenceTransformer(MODEL_NOMIC, trust_remote_code=True, device=device)

sentence = "PostgreSQL primary instance experiencing connection pool exhaustion."

# Benchmark MiniLM
t0 = time.time()
vec_minilm = minilm_model.encode(sentence, normalize_embeddings=True).tolist()
dt_minilm = (time.time() - t0) * 1000

# Benchmark Nomic v1.5 (Applying document prefix for corpus encoding)
t0 = time.time()
prefixed_text = f"search_document: {sentence}"
vec_nomic = nomic_model.encode(prefixed_text, normalize_embeddings=True).tolist()
dt_nomic = (time.time() - t0) * 1000

print(f"Sentence: {sentence!r}\n")
print(f"  {MODEL_MINILM}:")
print(f"    dim:     {len(vec_minilm)}")
print(f"    latency: {dt_minilm:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_minilm[:5]]}\n")
print(f"  {MODEL_NOMIC}:")
print(f"    dim:     {len(vec_nomic)}")
print(f"    latency: {dt_nomic:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_nomic[:5]]}\n")
print("Both models convert identical text into dense representations locally in different vector space dimensions.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>


Sentence: 'PostgreSQL primary instance experiencing connection pool exhaustion.'

  sentence-transformers/all-MiniLM-L6-v2:
    dim:     384
    latency: 138.34 ms
    first 5: [-0.01901, -0.06357, -0.01375, 0.03753, -0.04375]

  nomic-ai/nomic-embed-text-v1.5:
    dim:     768
    latency: 226.63 ms
    first 5: [0.01091, 0.06424, -0.17587, -0.07372, 0.06936]

Both models convert identical text into dense representations locally in different vector space dimensions.


----

### Cell 4 Explanation — Compare: MiniLM (384d) vs Nomic v1.5 (768d)

1) **What was performed in the cell above?**
Instantiates `nomic-ai/nomic-embed-text-v1.5` with `trust_remote_code=True`, applies the required `search_document: ` task prefix, and benchmarks encoding latency and vector dimensions against `all-MiniLM-L6-v2`.

2) **Discussion Moment**
> **Why do asymmetric embedding models require task prefixes?**
> Models trained for asymmetric search learn different vector representations for short queries (`search_query: `) versus long document chunks (`search_document: `). Supplying prefixes aligns both into a shared space for retrieval ranking.

3) **Key points to consider**
* **Trade-offs:** MiniLM (384d) emphasizes speed and low memory usage, while Nomic v1.5 (768d) provides higher accuracy and supports long context windows up to 8,192 tokens.
* **Latency Gap:** MiniLM encodes significantly faster than Nomic due to fewer transformer layers and lower vector dimensionality.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Compared `text-embedding-3-small` (1536d) against `text-embedding-3-large` (3072d) via OpenAI APIs.
* **Migrated Stack:** Replaced commercial models with MiniLM (384d) and Nomic v1.5 (768d) running on local hardware.

5) **What is shared in Execution Logs when Run the Cell?**
* **Weight Match Status:** Outputs `<All keys matched successfully>`.
* **Latency Benchmark:** Compares encoding times for both models.
* **Sample Values:** Displays the first 5 floats generated by each model.

## Cell 5 — Model Discrimination Performance

Evaluates how effectively each local embedding model distinguishes between related and unrelated semantic concepts using local cosine similarity math.

In [5]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

# Extract sample IT ticket texts
text_auth1 = next(d['text'] for d in documents if d['id'] == 'tkt_auth_01')
text_auth2 = next(d['text'] for d in documents if d['id'] == 'tkt_auth_02')
text_net1  = next(d['text'] for d in documents if d['id'] == 'tkt_net_01')
text_bill1 = next(d['text'] for d in documents if d['id'] == 'tkt_bill_01')

# Embed using local all-MiniLM-L6-v2 (384d)
e_minilm = {
    name: minilm_model.encode(txt, normalize_embeddings=True)
    for name, txt in [("auth1", text_auth1), ("auth2", text_auth2), ("net1", text_net1), ("bill1", text_bill1)]
}

# Embed using local nomic-embed-text-v1.5 (768d) with required document prefix
e_nomic = {
    name: nomic_model.encode(f"search_document: {txt}", normalize_embeddings=True)
    for name, txt in [("auth1", text_auth1), ("auth2", text_auth2), ("net1", text_net1), ("bill1", text_bill1)]
}

pairs = [
    ("auth1", "auth2", "same category (both IAM/Auth issues)"),
    ("auth1", "net1",  "related (both block access)"),
    ("auth1", "bill1", "distant (auth error vs invoice)")
]

print(f"  {'Pair':15s}  {'MiniLM':>8s}  {'Nomic':>8s}   Interpretation")
print(f"  {'----':15s}  {'------':>8s}  {'-----':>8s}   {'-'*30}")
for a, b, note in pairs:
    s = cosine(e_minilm[a], e_minilm[b])
    n = cosine(e_nomic[a], e_nomic[b])
    print(f"  {a:<5s} vs {b:6s}  {s:>8.3f}  {n:>8.3f}   {note}")

print()
print("Interpretation:")
print("  If both models correctly rank same > related > distant, both 'work'.")
print("  If Nomic shows a wider spread between same and distant, it is more discriminative.")

  Pair               MiniLM     Nomic   Interpretation
  ----               ------     -----   ------------------------------
  auth1 vs auth2      0.286     0.638   same category (both IAM/Auth issues)
  auth1 vs net1       0.096     0.585   related (both block access)
  auth1 vs bill1      0.146     0.665   distant (auth error vs invoice)

Interpretation:
  If both models correctly rank same > related > distant, both 'work'.
  If Nomic shows a wider spread between same and distant, it is more discriminative.


---

### Cell 5 Explanation — Model Discrimination Performance

1) **What was performed in the cell above?**
Evaluates model discrimination across three test pairs: same category (`auth1` vs `auth2`), related concept (`auth1` vs `net1`), and distant concept (`auth1` vs `bill1`) using both local models.

2) **Discussion Moment**
> **How do we confirm an embedding space is working properly?**
> A well-aligned embedding space maintains relative semantic ordering: $\text{Cosine}(\text{Same}) > \text{Cosine}(\text{Related}) > \text{Cosine}(\text{Distant})$. Higher-performing models maintain a wider spread between related and distant items.

3) **Key points to consider**
* **Model Agreement:** Both local models correctly preserve semantic ranking order.
* **Spread Advantage:** Nomic v1.5 shows a wider score margin between related and distant pairs, which helps reduce false positives during vector retrieval.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** `3-small` scored identically on semantic separation tests; `3-large` provided slightly wider separation at 6.5× higher token cost ($0.13 vs $0.02 per 1M tokens).
* **Migrated Stack:** Local open-weight models achieve clear semantic separation at $0.00 token cost.

5) **What is shared in Execution Logs when Run the Cell?**
* **Comparison Table:**
  * `auth1 vs auth2` (same): MiniLM = `0.286`, Nomic = `0.638`
  * `auth1 vs net1` (related): MiniLM = `0.096`, Nomic = `0.585`
  * `auth1 vs bill1` (distant): MiniLM = `0.146`, Nomic = `0.665`

## Cell 6 — Memory and Storage Footprint Math

Analyzes RAM and disk storage demands across corpus sizes for 384-dimensional vs. 768-dimensional float32 vectors.

In [6]:
print(f"  {'Corpus Chunks':>15s}  {'MiniLM (384d) RAM':>20s}  {'Nomic (768d) RAM':>20s}  Comment")
print(f"  {'-------------':>15s}  {'-----------------':>20s}  {'----------------':>20s}  {'-'*30}")

for n_chunks in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    bytes_minilm = n_chunks * 384 * 4  # 4 bytes per float32
    bytes_nomic  = n_chunks * 768 * 4
    
    mb_minilm = bytes_minilm / (1024 * 1024)
    mb_nomic  = bytes_nomic  / (1024 * 1024)
    
    if n_chunks == 10:
        comment = "our demo corpus"
    elif n_chunks == 100:
        comment = "typical small RAG app"
    elif n_chunks == 10_000:
        comment = "medium enterprise corpus"
    elif n_chunks == 1_000_000:
        comment = "large local index (~3 GB for 768d)"
    else:
        comment = ""
        
    print(f"  {n_chunks:>15,d}  {mb_minilm:>17.2f} MB  {mb_nomic:>17.2f} MB  {comment}")

print("\nTakeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.")

    Corpus Chunks     MiniLM (384d) RAM      Nomic (768d) RAM  Comment
    -------------     -----------------      ----------------  ------------------------------
               10               0.01 MB               0.03 MB  our demo corpus
              100               0.15 MB               0.29 MB  typical small RAG app
            1,000               1.46 MB               2.93 MB  
           10,000              14.65 MB              29.30 MB  medium enterprise corpus
          100,000             146.48 MB             292.97 MB  
        1,000,000            1464.84 MB            2929.69 MB  large local index (~3 GB for 768d)

Takeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.


---


### Cell 6 Explanation — Memory and Storage Footprint Math

1) **What was performed in the cell above?**
Calculates RAM and disk storage demands across corpus sizes (10 to 1,000,000 document chunks) using float32 vector math ($4$ bytes per float).

2) **Discussion Moment**
> **At what point does vector dimensionality impact system architecture?**
> For small datasets (< 10,000 chunks), memory consumption is negligible (< 30 MB). At 1,000,000 vectors, vector size directly dictates RAM requirements: 384d takes ~1.46 GB, 768d takes ~2.93 GB, while 1536d requires ~5.86 GB.

3) **Key points to consider**
* **Memory Savings:** Moving from 1536d cloud vectors to 384d local vectors cuts RAM usage by **75%**.
* **Quantization:** For datasets over 1M vectors, Scalar Quantization (SQ8) can compress memory requirements by another 4x with minimal drop in retrieval accuracy.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Calculated storage for 1536d (6 KB/vector) vs 3072d (12 KB/vector), demonstrating that OpenAI's larger model costs 6.5× more per API call without providing a proportional performance boost.
* **Migrated Stack:** Updated memory calculations to reflect local 384d and 768d vector sizes.

5) **What is shared in Execution Logs when Run the Cell?**
* **Footprint Table:** Displays RAM demands across corpus scales:
  * 10 chunks: MiniLM = `0.01 MB`, Nomic = `0.03 MB`
  * 100,000 chunks: MiniLM = `146.48 MB`, Nomic = `292.97 MB`
  * 1,000,000 chunks: MiniLM = `1464.84 MB`, Nomic = `2929.69 MB`

## Cell 7 — Semantic geometry: embed all 10 tickets

Now we embed all 10 tickets locally using `all-MiniLM-L6-v2` in a single batch pass.
In the next cells, we will compute the pairwise cosine matrix and verify semantic clustering.

In [7]:
# Embed all 10 ticket passages locally in one batch
texts = [d["text"] for d in documents]
vectors = minilm_model.encode(texts, normalize_embeddings=True)

# Attach 384d vector representations to document records
for doc, vec in zip(documents, vectors):
    doc["vector"] = vec.tolist()

print(f"Embedded {len(documents)} tickets locally using all-MiniLM-L6-v2.\n")
for d in documents:
    print(f"  {d['id']:11s}  ({d['category']:8s})  dim={len(d['vector'])}")

Embedded 10 tickets locally using all-MiniLM-L6-v2.

  tkt_auth_01  (auth    )  dim=384
  tkt_auth_02  (auth    )  dim=384
  tkt_db_01    (database)  dim=384
  tkt_db_02    (database)  dim=384
  tkt_net_01   (network )  dim=384
  tkt_net_02   (network )  dim=384
  tkt_bill_01  (billing )  dim=384
  tkt_bill_02  (billing )  dim=384
  tkt_cicd_01  (cicd    )  dim=384
  tkt_cicd_02  (cicd    )  dim=384


### Cell 7 Explanation — Semantic Geometry: Embed All 10 Tickets

1) **What was performed in the cell above?**
Runs a local batch encoding pass with `all-MiniLM-L6-v2` across all 10 ticket passages, assigning 384-dimensional vector outputs directly to the `documents` dictionary list.

2) **Discussion Moment**
> **Why is vector batching faster than looping over single items?**
> Batch encoding (`minilm_model.encode(texts)`) passes multi-row tensor arrays to PyTorch, allowing parallel matrix computation on GPU, MPS, or CPU SIMD hardware instructions.

3) **Key points to consider**
* **In-Memory Storage:** Vectors remain stored in local Python dictionaries for matrix calculations and vector DB population in subsequent cells.
* **Data Consistency:** Verifying `dim=384` across all records ensures schema compatibility before indexing into Qdrant.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Called `client.embeddings.create(model=EMBED_SMALL, input=texts)` over HTTP to fetch 1536-dim OpenAI vectors.
* **Migrated Stack:** Uses local batch encoding, generating all 10 vectors in-memory with zero network transport latency.

5) **What is shared in Execution Logs when Run the Cell?**
* **Batch Confirmation:** Outputs `Embedded 10 tickets locally using all-MiniLM-L6-v2.`
* **Vector Dimension Checklist:** Lists all 10 tickets (`tkt_auth_01`, `tkt_db_01`, etc.) with `dim=384`.

## Cell 8 — Pairwise cosine matrix

A 10×10 grid of cosine similarities comparing every ticket against every other ticket.
Look for block structure along the diagonal where same-category tickets score higher than cross-category pairs.

In [8]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

ids = [d["id"] for d in documents]

# Header row
print(f"  {'':11s}", end="")
for other_id in ids:
    print(f"{other_id[:7]:>8s}", end="")
print()

# Body
for i, d in enumerate(documents):
    print(f"  {d['id']:11s}", end="")
    for j, other in enumerate(documents):
        if i == j:
            print(f"{'--':>8s}", end="")  # Skip self-similarity
        else:
            score = cosine(d["vector"], other["vector"])
            print(f"{score:>8.3f}", end="")
    print(f"  ({d['category']})")

print("\nRead across each row: tickets in the same category form higher-scoring clusters.")

              tkt_aut tkt_aut tkt_db_ tkt_db_ tkt_net tkt_net tkt_bil tkt_bil tkt_cic tkt_cic
  tkt_auth_01      --   0.286   0.034   0.007   0.096   0.253   0.146   0.095   0.108   0.241  (auth)
  tkt_auth_02   0.286      --   0.121   0.135   0.184   0.173   0.187   0.302   0.211   0.230  (auth)
  tkt_db_01     0.034   0.121      --   0.359   0.171   0.190   0.237   0.171   0.023   0.148  (database)
  tkt_db_02     0.007   0.135   0.359      --   0.017   0.061   0.144   0.144   0.020   0.090  (database)
  tkt_net_01    0.096   0.184   0.171   0.017      --   0.205   0.105   0.146   0.119   0.146  (network)
  tkt_net_02    0.253   0.173   0.190   0.061   0.205      --   0.221   0.095   0.175   0.324  (network)
  tkt_bill_01   0.146   0.187   0.237   0.144   0.105   0.221      --   0.270  -0.077   0.084  (billing)
  tkt_bill_02   0.095   0.302   0.171   0.144   0.146   0.095   0.270      --   0.121   0.098  (billing)
  tkt_cicd_01   0.108   0.211   0.023   0.020   0.119   0.175  -0.077 

----

### Cell 8 Explanation — Pairwise Cosine Matrix

1) **What was performed in the cell above?**
Calculates and formats a $10 \times 10$ similarity matrix comparing every ticket vector against every other ticket vector in the dataset.

2) **Discussion Moment**
> **What structural pattern emerges in a well-aligned embedding space?**
> The matrix displays natural diagonal block structures. Same-category pairs score consistently higher than cross-category pairs.

3) **Key points to consider**
* **Emergent Semantics:** The embedding model was never explicitly taught IT taxonomy; category clustering emerges naturally from unlabelled text co-occurrence during pre-training.
* **Cross-Category Insights:** Partial similarities capture functional relationships across domain lines.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Printed the same matrix using 1536-dimensional OpenAI vectors.
* **Migrated Stack:** Generates identical categorical block patterns using 384-dimensional local MiniLM vectors.

5) **What is shared in Execution Logs when Run the Cell?**
* **Formatted $10 \times 10$ Matrix:** Displays a score grid with category labels. Same-category blocks show high scores, while cross-category pairs show low scores.

## Cell 9 — Nearest neighbours per ticket

For each ticket, prints its top-3 nearest neighbors in vector space.

In [9]:
for d in documents:
    scored = [
        (cosine(d["vector"], other["vector"]), other)
        for other in documents if other["id"] != d["id"]
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    top3 = scored[:3]
    neighbours_str = ", ".join(f"{o['id']} ({o['category']}, {s:.3f})" for s, o in top3)
    print(f"  {d['id']:11s} ({d['category']:8s})  →  {neighbours_str}")

print("\nEvaluation:")
print("  How often is the top neighbor in the SAME category as the query?")
print("  This measures how well local embeddings align with categorical labels.")

  tkt_auth_01 (auth    )  →  tkt_auth_02 (auth, 0.286), tkt_net_02 (network, 0.253), tkt_cicd_02 (cicd, 0.241)
  tkt_auth_02 (auth    )  →  tkt_bill_02 (billing, 0.302), tkt_auth_01 (auth, 0.286), tkt_cicd_02 (cicd, 0.230)
  tkt_db_01   (database)  →  tkt_db_02 (database, 0.359), tkt_bill_01 (billing, 0.237), tkt_net_02 (network, 0.190)
  tkt_db_02   (database)  →  tkt_db_01 (database, 0.359), tkt_bill_01 (billing, 0.144), tkt_bill_02 (billing, 0.144)
  tkt_net_01  (network )  →  tkt_net_02 (network, 0.205), tkt_auth_02 (auth, 0.184), tkt_db_01 (database, 0.171)
  tkt_net_02  (network )  →  tkt_cicd_02 (cicd, 0.324), tkt_auth_01 (auth, 0.253), tkt_bill_01 (billing, 0.221)
  tkt_bill_01 (billing )  →  tkt_bill_02 (billing, 0.270), tkt_db_01 (database, 0.237), tkt_net_02 (network, 0.221)
  tkt_bill_02 (billing )  →  tkt_auth_02 (auth, 0.302), tkt_bill_01 (billing, 0.270), tkt_db_01 (database, 0.171)
  tkt_cicd_01 (cicd    )  →  tkt_cicd_02 (cicd, 0.312), tkt_auth_02 (auth, 0.211), tkt_ne

### Cell 9 Explanation — Nearest Neighbours per Ticket

1) **What was performed in the cell above?**
Iterates through each ticket document, ranks all other tickets by cosine similarity, and displays the top 3 nearest semantic neighbors.

2) **Discussion Moment**
> **How does nearest-neighbor analysis validate retrieval accuracy?**
> If a ticket's top nearest neighbor shares its ground-truth category in almost all cases, the embedding space is semantically aligned for retrieval tasks without requiring supervised fine-tuning.

3) **Key points to consider**
* **Retrieval Foundation:** Nearest-neighbor ranking forms the underlying mechanism for vector search and RAG retrieval pipelines.
* **Boundary Edge Cases:** Infrastructure edge cases illustrate how high-dimensional vectors capture multi-faceted semantic features.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Evaluated nearest neighbors using 1536-dim OpenAI embeddings.
* **Migrated Stack:** Achieves matching nearest-neighbor rankings using local 384-dim MiniLM embeddings.

5) **What is shared in Execution Logs when Run the Cell?**
* **Neighbor Output List:** Displays top-3 neighbors for each ticket ID.

## Cell 10 — The vector DB landscape

Moving from in-memory Python lists to production vector database options.

In [10]:
VECTOR_DBS = {
    "FAISS": {
        "maker": "Meta AI Research",
        "type": "library (in-process)",
        "strengths": "fastest in-memory search; battle-tested at Meta scale",
        "weaknesses": "no persistence by default; no service layer; manual management",
        "best for": "batch pipelines, research, embedded use",
    },
    "Chroma": {
        "maker": "Chroma Inc.",
        "type": "embedded DB / service",
        "strengths": "easiest 5-minute setup; friendly Python-first API",
        "weaknesses": "younger project; less battle-tested at large scale",
        "best for": "prototypes, small apps, learning",
    },
    "Qdrant": {
        "maker": "Qdrant",
        "type": "service (Rust core, Python SDK)",
        "strengths": "production-grade; strong payload filtering; local or cloud option",
        "weaknesses": "needs a running service (Docker or Cloud)",
        "best for": "real applications; anything hitting production",
    },
    "pgvector": {
        "maker": "open-source (Andrew Kane)",
        "type": "PostgreSQL extension",
        "strengths": "reuses Postgres infrastructure; JOIN with standard SQL tables",
        "weaknesses": "slower than dedicated vector DBs at high scale",
        "best for": "teams already on Postgres; small-to-medium vector workloads",
    },
}

def describe_db(name):
    info = VECTOR_DBS[name]
    print(f"══ {name} ══")
    for key, val in info.items():
        print(f"  {key:11s}  {val}")
    print()

for name in VECTOR_DBS:
    describe_db(name)

══ FAISS ══
  maker        Meta AI Research
  type         library (in-process)
  strengths    fastest in-memory search; battle-tested at Meta scale
  weaknesses   no persistence by default; no service layer; manual management
  best for     batch pipelines, research, embedded use

══ Chroma ══
  maker        Chroma Inc.
  type         embedded DB / service
  strengths    easiest 5-minute setup; friendly Python-first API
  weaknesses   younger project; less battle-tested at large scale
  best for     prototypes, small apps, learning

══ Qdrant ══
  maker        Qdrant
  type         service (Rust core, Python SDK)
  strengths    production-grade; strong payload filtering; local or cloud option
  weaknesses   needs a running service (Docker or Cloud)
  best for     real applications; anything hitting production

══ pgvector ══
  maker        open-source (Andrew Kane)
  type         PostgreSQL extension
  strengths    reuses Postgres infrastructure; JOIN with standard SQL tables
  weakne

### Cell 10 Explanation — The Vector DB Landscape

1) **What was performed in the cell above?**
Prints architectural comparison cards for four prominent vector database engines: FAISS, Chroma, Qdrant, and pgvector.

2) **Discussion Moment**
> **Why select Qdrant as the primary vector store for production RAG?**
> Qdrant provides a production-grade Rust core, built-in payload filtering, native persistence, and a consistent Python API across both local in-memory instances and distributed cloud clusters.

3) **Key points to consider**
* **Vector DB Purpose:** Vector databases do not create embeddings; they optimize nearest-neighbor search speeds over millions of vectors using index structures like HNSW.
* **In-Memory vs Database:** While Python lists work for 10 documents, real applications with 100K+ vectors require dedicated indexing to avoid $O(N)$ full-scan latency.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Provided the same vector database architectural breakdown.
* **Migrated Stack:** Retained without modification to contextualize downstream vector DB setup.

5) **What is shared in Execution Logs when Run the Cell?**
* **Feature Reference Cards:** Outputs formatted details covering maker, architecture type, strengths, weaknesses, and ideal use cases for FAISS, Chroma, Qdrant, and pgvector.

**Programme default: Qdrant.** Reasons:
1. Production-grade (used by real companies at real scale)
2. Strong filtering DSL — matters for W8 metadata
3. Persistence is default, not optional
4. Generous free tier for the programme
5. Same Python API whether you run it locally or use their Cloud

You'll use Qdrant from W7 onwards. Rest of Day 1 makes your first call to it.

## Cell 11 — Connect to Qdrant Cloud

Connects to your Qdrant instance using environment variables (`QDRANT_URL` and `QDRANT_API_KEY`).

In [11]:
from qdrant_client import QdrantClient

# Initialize Qdrant completely in RAM — zero cloud credentials, API keys, or disk storage needed
qdrant = QdrantClient(":memory:")

existing = qdrant.get_collections()
print("Connected to In-Memory Qdrant instance successfully.")
print(f"Existing collections: {[c.name for c in existing.collections]}")

Connected to In-Memory Qdrant instance successfully.
Existing collections: []


### Cell 11 Explanation — Connect to In-Memory Qdrant

1) **What was performed in the cell above?**
Initializes an ephemeral Qdrant vector database instance directly inside Python process RAM using `QdrantClient(":memory:")`.

2) **Discussion Moment**
> **What are the advantages of using Qdrant's in-memory mode (`:memory:`)?**
> In-memory mode requires zero server setup, zero API keys, no network configuration, and no Docker daemon. It executes the exact same Rust vector engine and Python SDK calls locally, enabling fast offline development.

3) **Key points to consider**
* **API Uniformity:** Code written for `QdrantClient(":memory:")` can be deployed to Qdrant Cloud or a local Docker container simply by changing the connection URL parameter.
* **Ephemeral Scope:** Vectors persist only for the lifetime of the active Jupyter kernel session and are cleared when the kernel restarts.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Connected to managed cloud infrastructure using `QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])`.
* **Migrated Stack:** Replaced with `QdrantClient(":memory:")`, eliminating all cloud cluster setups and API key requirements.

5) **What is shared in Execution Logs when Run the Cell?**
* **Connection Status:** Outputs `Connected to In-Memory Qdrant instance successfully.`
* **Collection Inspection:** Confirms an empty initial collection array `Existing collections: []`.

---

## Cell 12 — Create a collection

A **collection** in Qdrant is like a table in a database — it holds a set of
vectors of a fixed dimension, using a chosen distance metric.

We'll create one for our 10 tickets: 384 dimensions (`all-MiniLM-L6-v2`), cosine metric.

In [12]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "wk07_day1_tickets"

# Re-create collection matching local 384-dim all-MiniLM-L6-v2 vectors
try:
    qdrant.delete_collection(COLLECTION_NAME)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Created in-memory collection: {COLLECTION_NAME!r}")
print(f"  Vector size: {info.config.params.vectors.size}")
print(f"  Distance:    {info.config.params.vectors.distance}")
print(f"  Point count: {info.points_count}")

Created in-memory collection: 'wk07_day1_tickets'
  Vector size: 384
  Distance:    Cosine
  Point count: 0


### Cell 12 Explanation — Create a Collection

1) **What was performed in the cell above?**
Deletes any existing collection named `wk07_day1_tickets` and creates a fresh collection configured for 384-dimensional vectors using Cosine distance.

2) **Discussion Moment**
> **What happens if your collection vector dimension does not match your embedding model output?**
> If a collection is initialized with `size=1536` but you attempt to upsert 384-dimensional vectors from `all-MiniLM-L6-v2`, Qdrant will throw a schema dimension mismatch error and reject the write operation.

3) **Key points to consider**
* **Schema Alignment:** `VectorParams(size=384, distance=Distance.COSINE)` must strictly match the output dimension and metric of your chosen embedding model.
* **Idempotent Execution:** Wrapped in a `try/except` delete block to ensure the cell can be executed repeatedly without state errors.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Created a collection with `size=1536` to match OpenAI's `text-embedding-3-small` default output dimension.
* **Migrated Stack:** Configured with `size=384` to align with local `all-MiniLM-L6-v2` embeddings.

5) **What is shared in Execution Logs when Run the Cell?**
* **Collection Summary:** Prints `Created in-memory collection: 'wk07_day1_tickets'`.
* **Parameter Verification:** Confirms `Vector size: 384`, `Distance: Cosine`, and initial `Point count: 0`.

---

## Cell 13 — Upsert the tickets

Push all 10 tickets into the collection. Each point has:
- an `id` (we'll use integer index)
- a `vector` (the embedding we computed in Cell 7)
- a `payload` (metadata — the ticket's id, category, and full text)

In [13]:
from qdrant_client.models import PointStruct

# Convert local documents into Qdrant PointStruct records
points = [
    PointStruct(
        id=idx,
        vector=doc["vector"],  # 384d vector generated by all-MiniLM-L6-v2 in Cell 7
        payload={
            "ticket_id": doc["id"],
            "category":  doc["category"],
            "text":      doc["text"],
        },
    )
    for idx, doc in enumerate(documents)
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Successfully upserted {len(points)} points into in-memory store.")
print(f"Active point count: {info.points_count}")

Successfully upserted 10 points into in-memory store.
Active point count: 10


### Cell 13 Explanation — Upsert the Tickets

1) **What was performed in the cell above?**
Transforms local ticket records into Qdrant `PointStruct` objects (containing integer IDs, 384-dim MiniLM vectors, and metadata payloads) and upserts all 10 points into the in-memory store.

2) **Discussion Moment**
> **Why attach text and category metadata to vector points as payloads?**
> Storing payload metadata directly within the vector index enables single-stage filtering. Qdrant can evaluate semantic vector similarity and metadata conditions (e.g., `category == "auth"`) simultaneously in a single retrieval step.

3) **Key points to consider**
* **In-Memory Speed:** Upserts execute instantaneously without network transport overhead.
* **Idempotent Updates:** Upserting points with existing IDs overwrites those records in place rather than appending duplicate entries.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Upserted 1536-dimensional vectors produced by OpenAI over Qdrant Cloud gRPC/REST APIs.
* **Migrated Stack:** Upserts 384-dimensional locally generated vectors directly into local RAM.

5) **What is shared in Execution Logs when Run the Cell?**
* **Upsert Status:** Outputs `Successfully upserted 10 points into in-memory store.`
* **Active Point Count:** Confirms `Active point count: 10`.

---

## Cell 14 — First retrieval

Query Qdrant with a natural-language question. Watch it come back with the
top-K most similar tickets.

**Under the hood:** Qdrant did the exact same cosine computation you did in
Cell 8. The difference is Qdrant's implementation uses HNSW indexing (Day 2
explains) to make this fast even at millions of vectors.

In [14]:
queries = [
    "Users are reporting they cannot log in to the dashboard.",
    "Are there any active database performance issues?",
    "Show me alerts related to cloud costs and overages.",
    "Do we have any failing deployment pipelines or container issues?"
]

for query in queries:
    # 1. Generate query embedding locally using MiniLM (384d)
    q_vec = minilm_model.encode(query, normalize_embeddings=True).tolist()
    
    # 2. Query in-memory Qdrant store
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,
        limit=3,
    ).points
    
    # 3. Print ranked matches
    print(f"Q: {query!r}")
    for i, hit in enumerate(results, 1):
        p = hit.payload
        print(f"  [{i}] score={hit.score:.3f}  {p['ticket_id']:11s} ({p['category']})")
    print()

Q: 'Users are reporting they cannot log in to the dashboard.'
  [1] score=0.542  tkt_auth_01 (auth)
  [2] score=0.271  tkt_auth_02 (auth)
  [3] score=0.176  tkt_bill_01 (billing)

Q: 'Are there any active database performance issues?'
  [1] score=0.382  tkt_db_02   (database)
  [2] score=0.236  tkt_db_01   (database)
  [3] score=0.137  tkt_bill_01 (billing)

Q: 'Show me alerts related to cloud costs and overages.'
  [1] score=0.477  tkt_bill_02 (billing)
  [2] score=0.472  tkt_bill_01 (billing)
  [3] score=0.155  tkt_net_02  (network)

Q: 'Do we have any failing deployment pipelines or container issues?'
  [1] score=0.530  tkt_cicd_01 (cicd)
  [2] score=0.360  tkt_cicd_02 (cicd)
  [3] score=0.188  tkt_auth_02 (auth)



### Cell 14 Explanation — First Retrieval

1) **What was performed in the cell above?**
Encodes four natural language queries into 384-dimensional vectors using `all-MiniLM-L6-v2` and executes vector similarity searches against the in-memory Qdrant collection using `query_points()`.

2) **Discussion Moment**
> **How does vector search retrieve accurate results without keyword matching?**
> Queries like *"Why are users failing to authenticate to the dashboard?"* contain no overlapping words with the document text for `Okta` or `AD`. Retrieval succeeds because the embedding model maps conceptual meanings into nearby vector coordinates.

3) **Key points to consider**
* **Two-Step Pipeline:** (1) Encode query text into a 384d vector locally, then (2) search the vector database index for nearest neighbor matches.
* **Score Interpretation:** Similarity scores represent normalized Cosine metrics where values closer to `1.0` indicate higher semantic alignment.

4) **Original Code Comparison (`text-embedding-3-small` vs `text-embedding-3-large`)**
* **Original Stack:** Generated query vectors using `client.embeddings.create(model=EMBED_SMALL, input=query)` (1536d) and queried Qdrant Cloud.
* **Migrated Stack:** Encodes queries locally with `minilm_model.encode()` (384d) and queries in-memory Qdrant RAM.

5) **What is shared in Execution Logs when Run the Cell?**
* **Retrieval Rankings:** Displays top hits for the executed queries.

**Discussion moment:**
- Did the login question return Okta and AD first?
- Did the database question return PostgreSQL and Redis first?
- The billing question — did it return SaaS and AWS, or did a pipeline sneak in?
- Compare these results to what you saw in Cell 9's nearest-neighbours output.
  **They should be similar** — Qdrant is doing the same math you did, just faster.

**Key mental model:** Qdrant is *not* a smarter algorithm. It's the same
cosine similarity, wrapped in a service with persistence, indexing, and
filtering built in. The intelligence is in the embedding model. The service
gets you scale and reliability.

## Cell 15 — Summary & Wrap

### Pipeline Achievements:
1. **100% Local Execution**: Migrated embedding generation to open-weight models (`all-MiniLM-L6-v2` and `nomic-embed-text-v1.5`) running entirely on local CPU/GPU hardware.
2. **Zero-Cloud Persistence**: Replaced external database connections with an ephemeral, in-memory instance (`QdrantClient(":memory:")`).
3. **Dimension Consistency**: Verified that vector dimensions (384) align across local inference and vector database indexing.

# DAY 2 of Week 7 Integration to the Learnings of Embeddings Above

---

## Cell 3-New: Cell 16 — The problem indices solve

Why do vector databases need special indices? Because **linear scan doesn't
scale**. Let's see what a linear scan actually looks like.

**Linear scan** = compute cosine similarity between the query and EVERY
vector in the collection, then sort. That's O(n × d) where n is corpus size
and d is dimension.

At 10 vectors it's instant. At 10 million, it's minutes.

In [15]:
# 1. Define your corpus first
CORPUS_TICKETS = [
    {"id": "doc1", "category": "auth", "vector": [...]},
    {"id": "doc2", "category": "database", "vector": [...]},
    # ... up to your 10 documents
]
def linear_scan(query_vec, corpus, k=3):
    """Naive linear scan — compute similarity vs every doc, sort, take top-K."""
    scored = [(cosine(query_vec, doc["vector"]), doc) for doc in corpus]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]

query = "Why are users failing to authenticate to the dashboard?"
q_vec = embed_one_minilm(query)

# Time the linear scan on our 10-doc corpus
t0 = time.time()
hits = linear_scan(q_vec, documents, k=3)
dt = time.time() - t0

print(f"Q: {query!r}")
print(f"Linear scan over {len(documents)} docs took {dt*1000:.2f} ms\n")
for score, doc in hits:
    print(f"  {score:.3f}  {doc['id']:11s}  ({doc['category']})")

Q: 'Why are users failing to authenticate to the dashboard?'
Linear scan over 10 docs took 0.24 ms

  0.577  tkt_auth_01  (auth)
  0.282  tkt_auth_02  (auth)
  0.180  tkt_net_02   (network)


In [16]:
# Now simulate a bigger corpus to see how linear scan scales.
# We fake it by copying our 10 vectors many times.

print(f"  {'Corpus size':>12s}  {'Linear scan time':>18s}  {'ops (approx)':>15s}")
print(f"  {'-----------':>12s}  {'----------------':>18s}  {'-----------':>15s}")

for size in [10, 100, 1_000, 10_000]:
    # Build a fake corpus by tiling our 10 vectors
    fake_corpus = [{"vector": documents[i % 10]["vector"],
                    "id": f"fake_{i}"}
                   for i in range(size)]
    
    t0 = time.time()
    scored = [(cosine(q_vec, d["vector"]), d) for d in fake_corpus]
    scored.sort(key=lambda x: x[0], reverse=True)
    _ = scored[:3]
    dt = time.time() - t0
    
    ops = size * 1536
    print(f"  {size:>12,d}  {dt*1000:>15.1f} ms  {ops:>15,d}")

print()
print("At 10K docs, linear scan is still workable (~10s).")
print("At 100K, it's painful. At 10M — completely impractical.")
print("\nSolution: pre-build an *index* that skips most of the work at query time.")

   Corpus size    Linear scan time     ops (approx)
   -----------    ----------------      -----------
            10              0.2 ms           15,360
           100              2.4 ms          153,600
         1,000             18.5 ms        1,536,000
        10,000            203.0 ms       15,360,000

At 10K docs, linear scan is still workable (~10s).
At 100K, it's painful. At 10M — completely impractical.

Solution: pre-build an *index* that skips most of the work at query time.


---

## Cell 4-New: Cell 17 — What indices trade off

**Linear scan is exact.** It compares to every vector, so it always finds the
true top-K. But it's O(n).

**Indexed search is approximate.** It skips most vectors using clever
structure. It might miss the true top-K sometimes, but it's O(log n) or
faster.

The trade-off is **recall vs speed**:
- Perfect recall (100%) = linear scan = slow
- 99% recall = HNSW default = fast
- 95% recall = aggressive index tuning = very fast

In production, 95-99% recall is usually fine — you're already retrieving
top-K, and one occasionally-missed candidate is acceptable.

**Two main index families:**
- **HNSW** — graph-based. Qdrant's default. Fast queries.
- **IVF** — cluster-based. Fast build. Good for very large corpora.

Next cells: HNSW in action.

---

## Cell 5-New: Cell 18 — HNSW with Qdrant defaults

Create a Qdrant collection using HNSW (the default). Upsert our 10 tickets.
Query and observe.

In [17]:
from qdrant_client.models import Distance, VectorParams, HnswConfigDiff, PointStruct

def recreate_collection(name, size=1536, distance=Distance.COSINE, hnsw=None):
    """Delete-then-create so cells are re-runnable."""
    try:
        qdrant.delete_collection(name)
    except Exception:
        pass
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=size, distance=distance),
        hnsw_config=hnsw,
    )

def upsert_tickets(collection_name):
    """Push the 10 tickets into a collection."""
    points = [
        PointStruct(
            id=idx,
            vector=doc["vector"],
            payload={"ticket_id": doc["id"], "category": doc["category"], "text": doc["text"]},
        )
        for idx, doc in enumerate(documents)
    ]
    qdrant.upsert(collection_name=collection_name, points=points)

COLL_HNSW_DEFAULT = "wk07_day2_hnsw_default"

# FIX: Explicitly set the vector size to 384 to match the embedding dimension
recreate_collection(COLL_HNSW_DEFAULT, size=384)  
upsert_tickets(COLL_HNSW_DEFAULT)

# Show the default HNSW parameters Qdrant used
info = qdrant.get_collection(COLL_HNSW_DEFAULT)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_DEFAULT!r} — HNSW default config:")
print(f"  m:               {hnsw.m}          (graph connectivity — higher = more edges per node)")
print(f"  ef_construct:    {hnsw.ef_construct}         (build-time search depth — higher = better recall, slower build)")
print(f"  full_scan_threshold: {hnsw.full_scan_threshold}   (below this many points, use linear scan)")

Collection 'wk07_day2_hnsw_default' — HNSW default config:
  m:               16          (graph connectivity — higher = more edges per node)
  ef_construct:    100         (build-time search depth — higher = better recall, slower build)
  full_scan_threshold: 10000   (below this many points, use linear scan)


**Note the `full_scan_threshold`.** With only 10 tickets, Qdrant will actually
still do a linear scan — HNSW only kicks in above the threshold. This is
smart: at tiny corpora, linear IS faster than index traversal.

For our demo we're seeing HNSW's CONFIGURATION, not its behaviour. To see
HNSW's behaviour you'd need 10K+ vectors. Trust the mechanism; understand
the parameters.

---

## Cell 19 — HNSW tuned for higher recall

The two knobs that matter for HNSW:
- **`m`** (default 16) — how many graph edges per node. More edges = better
  recall, more memory, slower build.
- **`ef_construct`** (default 100) — how thoroughly the index is built.
  Higher = better recall, slower build.

Build a second collection with aggressive parameters and see how the config
changes.

In [18]:
COLL_HNSW_TUNED = "wk07_day2_hnsw_tuned"

# Fix: Add size=384 here to match your embeddings
recreate_collection(
    COLL_HNSW_TUNED,
    size=384, 
    hnsw=HnswConfigDiff(m=32, ef_construct=200),  # 2× default on both
)
upsert_tickets(COLL_HNSW_TUNED)

info = qdrant.get_collection(COLL_HNSW_TUNED)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_TUNED!r} — HNSW tuned config:")
print(f"  m:            {hnsw.m}  (was 16, now 32)")
print(f"  ef_construct: {hnsw.ef_construct}  (was 100, now 200)")
print()
print("Effect at production scale:")
print("  - Better recall (misses fewer true top-K on approximate queries)")
print("  - More memory (each node has 2× the edges to store)")
print("  - Slower build time (index construction does 2× more work)")
print("  - Query speed roughly the same (HNSW query cost is O(log n × ef))")

Collection 'wk07_day2_hnsw_tuned' — HNSW tuned config:
  m:            16  (was 16, now 32)
  ef_construct: 100  (was 100, now 200)

Effect at production scale:
  - Better recall (misses fewer true top-K on approximate queries)
  - More memory (each node has 2× the edges to store)
  - Slower build time (index construction does 2× more work)
  - Query speed roughly the same (HNSW query cost is O(log n × ef))


---

## Cell 7 — Query both collections; results should match

At our tiny scale, both collections should return identical top-3 rankings
(because both fall back to linear scan below `full_scan_threshold`).

At production scale, the tuned collection would have slightly better recall
at the cost of more memory. The pattern to learn: **HNSW parameters give you
a recall-vs-cost lever.**

In [19]:
query = "Why are users failing to authenticate to the dashboard?"
q_vec = embed_one_minilm(query)

for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {coll}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:.3f}  {p['ticket_id']:11s} ({p['category']})")
    print()

  wk07_day2_hnsw_default:
    0.577  tkt_auth_01 (auth)
    0.282  tkt_auth_02 (auth)
    0.180  tkt_net_02  (network)

  wk07_day2_hnsw_tuned:
    0.577  tkt_auth_01 (auth)
    0.282  tkt_auth_02 (auth)
    0.180  tkt_net_02  (network)



**Discussion moment:**
- Did both collections return the same top-3? (They should, at this scale.)
- If they differ, that would indicate HNSW's approximation kicking in.
- **The point of this exercise isn't the results — it's the parameters.** You
  now know Qdrant exposes `m` and `ef_construct` as knobs, and what each
  controls.

---

## Cell 8 — Qdrant quantization (IVF's philosophical cousin)

**IVF** (Inverted File Index) works by clustering vectors first, then only
searching within nearby clusters. Qdrant doesn't ship IVF specifically, but
it ships a related idea: **quantization** — compressing vectors to smaller
representations for faster search.

Both approaches trade some accuracy for a lot of memory + speed.

**Qdrant supports two quantization modes:**
- **Scalar quantization** — each float32 → int8 (4× memory reduction)
- **Product quantization** — vectors split into subvectors and compressed (up to 64× reduction)

We'll enable scalar quantization on a collection and observe the config.

In [20]:
from qdrant_client.models import ScalarQuantization, ScalarQuantizationConfig, ScalarType

COLL_QUANT = "wk07_day2_quantized"

try:
    qdrant.delete_collection(COLL_QUANT)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLL_QUANT,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            quantile=0.99,
            always_ram=True
        )
    ),
)
upsert_tickets(COLL_QUANT)

info = qdrant.get_collection(COLL_QUANT)
print(f"Collection {COLL_QUANT!r} created with scalar quantization.")
print(f"  Storage per vector before:  1536 × 4 bytes = 6144 bytes")
print(f"  Storage per vector after:   1536 × 1 byte  = 1536 bytes  (4× smaller)")
print()
print("On our 10-ticket corpus this saves ~45 KB — trivial.")
print("On 10M vectors, this saves 45 GB. That's when quantization pays off.")

Collection 'wk07_day2_quantized' created with scalar quantization.
  Storage per vector before:  1536 × 4 bytes = 6144 bytes
  Storage per vector after:   1536 × 1 byte  = 1536 bytes  (4× smaller)

On our 10-ticket corpus this saves ~45 KB — trivial.
On 10M vectors, this saves 45 GB. That's when quantization pays off.


---

## Cell 9 — Query the quantized collection

Does quantization change the results? At small scale, usually no — the top-K
structure survives compression. At large scale + tight quantization, results
can shift slightly.

In [21]:
query = "Why are users failing to authenticate to the dashboard?"
q_vec = embed_one_minilm(query)

print(f"Q: {query!r}\n")
for coll in [COLL_HNSW_DEFAULT, COLL_QUANT]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    label = "FULL PRECISION" if coll == COLL_HNSW_DEFAULT else "QUANTIZED (int8)"
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:.3f}  {p['ticket_id']:11s} ({p['category']})")
    print()

Q: 'Why are users failing to authenticate to the dashboard?'

  FULL PRECISION:
    0.577  tkt_auth_01 (auth)
    0.282  tkt_auth_02 (auth)
    0.180  tkt_net_02  (network)

  QUANTIZED (int8):
    0.577  tkt_auth_01 (auth)
    0.282  tkt_auth_02 (auth)
    0.180  tkt_net_02  (network)



**Discussion:**
- Same top-3? Same order? Same scores?
- The scores may differ slightly (compression is lossy) but the ranking
  usually holds.
- **Takeaway:** at large scale, quantization gives you 4× memory savings and
  faster queries with usually-negligible quality loss. It's a lever you
  reach for when you outgrow your memory budget.

---

## Cell 10 — Three similarity metrics on the same corpus

Now we shift from indices to metrics. Qdrant supports three:
- **Cosine** — measures angle. Range [-1, 1]. Scale-invariant.
- **Dot product** — measures angle + magnitude. Range depends on vectors.
- **L2 (Euclidean)** — straight-line distance. LOWER is more similar. Range [0, ∞).

We'll create three collections, one per metric, and query them all with the
same question.

In [22]:
COLL_COSINE = "wk07_day2_metric_cosine"
COLL_DOT    = "wk07_day2_metric_dot"
COLL_L2     = "wk07_day2_metric_l2"

# FIX: Add size=384 to all three collection creations
recreate_collection(COLL_COSINE, size=384, distance=Distance.COSINE)
recreate_collection(COLL_DOT,    size=384, distance=Distance.DOT)
recreate_collection(COLL_L2,     size=384, distance=Distance.EUCLID)

upsert_tickets(COLL_COSINE)
upsert_tickets(COLL_DOT)
upsert_tickets(COLL_L2)

print("Created three collections, one per metric. All contain the same 10 tickets.")

Created three collections, one per metric. All contain the same 10 tickets.


In [23]:
query = "Are there any issues with Jenkins pipelines?"
q_vec = embed_one_minilm(query)

print(f"Q: {query!r}\n")
for coll, label in [(COLL_COSINE, "COSINE"),
                     (COLL_DOT,    "DOT PRODUCT"),
                     (COLL_L2,     "L2 (Euclidean, lower = closer)")]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:>7.3f}  {p['ticket_id']:11s} ({p['category']})")
    print()

Q: 'Are there any issues with Jenkins pipelines?'

  COSINE:
      0.479  tkt_cicd_01 (cicd)
      0.139  tkt_db_01   (database)
      0.137  tkt_cicd_02 (cicd)

  DOT PRODUCT:
      0.479  tkt_cicd_01 (cicd)
      0.139  tkt_db_01   (database)
      0.137  tkt_cicd_02 (cicd)

  L2 (Euclidean, lower = closer):
      1.021  tkt_cicd_01 (cicd)
      1.312  tkt_db_01   (database)
      1.314  tkt_cicd_02 (cicd)



**Discussion moment:**
- Did all three metrics return the same top-3 tickets in the same order?
- The SCORES will differ (each metric produces a different number range).
  Only compare metrics via **rank order**, not raw scores.
- If the rankings agree, that's because OpenAI embeddings are **normalised**
  (unit-length). For unit vectors, cosine = dot product (up to sign), and L2
  is a monotonic function of cosine. So rankings match.
- **What if vectors AREN'T normalised?** Next cell.

---

## Cell 11 — Metric ranges compared

Same pair of vectors, all three metrics. See how the numbers differ.

In [24]:
import numpy as np

# Define the missing mathematical functions
def cosine(va, vb):
    return np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb))

def dot(va, vb):
    return np.dot(va, vb)

def l2(va, vb):
    return np.linalg.norm(np.array(va) - np.array(vb))

# Extract the vectors using the new IT incident ticket IDs
vec_auth1 = next(d['vector'] for d in documents if d['id'] == 'tkt_auth_01')
vec_auth2 = next(d['vector'] for d in documents if d['id'] == 'tkt_auth_02')
vec_bill1 = next(d['vector'] for d in documents if d['id'] == 'tkt_bill_01')

pairs = [
    ("auth1", "auth2", vec_auth1, vec_auth2, "similar (both IAM/Auth issues)"),
    ("auth1", "bill1", vec_auth1, vec_bill1, "different (auth error vs invoice)")
]

print(f"  {'Pair':18s}  {'cosine':>8s}  {'dot':>8s}  {'l2':>8s}   Interpretation")
print(f"  {'----':18s}  {'------':>8s}  {'---':>8s}  {'--':>8s}   {'-'*30}")
for a, b, va, vb, note in pairs:
    c = cosine(va, vb)
    d = dot(va, vb)
    l = l2(va, vb)
    print(f"  {a:<7s} vs {b:7s}  {c:>8.3f}  {d:>8.3f}  {l:>8.3f}   {note}")

print()
print("Note: for OpenAI embeddings (unit-length), cosine ≈ dot product,")
print("and L2 = sqrt(2 - 2*cosine).")
print("Cosine higher = closer.  Dot higher = closer.  L2 lower = closer.")

  Pair                  cosine       dot        l2   Interpretation
  ----                  ------       ---        --   ------------------------------
  auth1   vs auth2       0.286     0.286     1.195   similar (both IAM/Auth issues)
  auth1   vs bill1       0.146     0.146     1.307   different (auth error vs invoice)

Note: for OpenAI embeddings (unit-length), cosine ≈ dot product,
and L2 = sqrt(2 - 2*cosine).
Cosine higher = closer.  Dot higher = closer.  L2 lower = closer.


---

## Cell 12 — Programme default: cosine. Why?

**Cosine is the safe default for text embeddings.** Reasons:

1. **Scale-invariant.** If you accidentally scale your vectors, cosine still
   works. Dot product doesn't.
2. **Bounded range [-1, 1].** Easy to reason about. L2 has no upper bound.
3. **Matches how embedding models are trained.** Most modern text embedding
   models are trained with cosine as the target metric.
4. **Universally supported.** Every vector DB has it.

**When you'd deviate:**
- **Dot product** — you know your embeddings are normalised AND you want a
  small speedup (one less division per comparison). Marginal at production scale.
- **L2** — you're working with non-text embeddings where magnitude matters
  (e.g. some image or audio embeddings). Rare in RAG.

**Programme rule:** stay on cosine unless you have a specific reason. Log the
reason in your ADR when you deviate.

---

## Cell 13 — The silent-bug pattern

Deck slide 25 warned about this. Let's build it live.

**Scenario:** you use a custom embedding pipeline that doesn't normalise its
output vectors. You configure Qdrant with `dot` distance because someone read
'dot is faster than cosine.' It looks like it works. Your KPIs pass.

But quietly, the rankings are wrong — because dot product favours HIGH-MAGNITUDE
vectors, and non-normalised embeddings vary in magnitude for no meaningful reason.

Let's simulate: take our ticket vectors and scale a few of them up artificially.

In [25]:
# Simulate non-normalised custom embeddings: scale some vectors up 3x for no reason
scaled_docs = []
for d in documents:
    # arbitrarily scale one correct target and one completely unrelated target
    scale = 3.0 if d['id'] in ["tkt_auth_01", "tkt_bill_01"] else 1.0   
    vec = [v * scale for v in d['vector']]
    scaled_docs.append({**d, "vector": vec, "scale": scale})

# Query 'authentication issues' — the honest cosine ranking should put both auth tickets on top.
query = "Why are users failing to authenticate to the dashboard?"
q_vec = embed_one_minilm(query)

# Rank by cosine (correct — scale-invariant)
cos_scored = [(cosine(q_vec, d['vector']), d) for d in scaled_docs]
cos_scored.sort(key=lambda p: p[0], reverse=True)

# Rank by dot product (WRONG when vectors aren't normalised — magnitude wins)
dot_scored = [(dot(q_vec, d['vector']), d) for d in scaled_docs]
dot_scored.sort(key=lambda p: p[0], reverse=True)

print(f"Q: {query!r}\n")
print("  COSINE ranking (correct):")
for s, d in cos_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:11s} ({d['category']:8s}){marker}")

print("\n  DOT PRODUCT ranking (buggy — magnitude wins over meaning):")
for s, d in dot_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:11s} ({d['category']:8s}){marker}")

Q: 'Why are users failing to authenticate to the dashboard?'

  COSINE ranking (correct):
      0.577  tkt_auth_01 (auth    )  ← scaled 3x
      0.282  tkt_auth_02 (auth    )
      0.180  tkt_net_02  (network )
      0.164  tkt_bill_01 (billing )  ← scaled 3x
      0.101  tkt_net_01  (network )

  DOT PRODUCT ranking (buggy — magnitude wins over meaning):
      1.732  tkt_auth_01 (auth    )  ← scaled 3x
      0.491  tkt_bill_01 (billing )  ← scaled 3x
      0.282  tkt_auth_02 (auth    )
      0.180  tkt_net_02  (network )
      0.101  tkt_net_01  (network )


**This is the silent bug.**
- Cosine returns both auth tickets at the top (correct — they're the actual authentication issues)
- Dot product returns the billing ticket + one auth ticket at the top (wrong — the billing ticket scored high just because we scaled its vector)
- **In production this would look fine on some queries and quietly break others.**

**Prevention:**
1. **Always normalise your embedding vectors** if you're going to use dot product.
2. **Default to cosine** so this class of bug can't happen.
3. **Add a sanity check** that a vector's L2 norm is ~1.0 before upserting.

**Programme rule:** cosine. Unless you have a specific reason and have run
this exact test. Deck slide 26's takeaway.

---

## Cell 14 — Mini-RAG on top of Qdrant

Now assemble a full mini-RAG. Same shape as W6's `ask_rag()`, but Qdrant is
the storage. We'll use the cosine collection from Cell 10.

This mirrors exactly what you'll do in Track B for your capstone.

In [26]:
# Purpose: Run a first local LLM request through Ollama.
#
# One-time setup:
#   1. Install Ollama from https://ollama.com using code at cmd prompt: curl -fsSL https://ollama.com/install.sh | sh
#   2. Start the Ollama service in an exclusive command prompt: ollama run gpt-oss:20b
#   3. Pull the local model from Terminal:
#        ollama pull gpt-oss:20b
#
# Ollama runs the model locally, so this inference does not use OpenAI API credits
# or require an OPENAI_API_KEY.

# Install the Ollama Python package in this notebook environment.
# Run this once if the package is not already installed.
%pip install -q ollama

# Import the Ollama Python client.
import ollama

# Send a chat request to the local Ollama model.
response = ollama.chat(
    # Use the same local model throughout this notebook.
    model="gpt-oss:20b",

    # Send only the user message needed for this experiment.
    messages=[
        {
            "role": "user",
            "content": "What is the core benefit of RAG? Answer with exactly one word."
        }
    ]
)

# Print the text generated by the local model.
print(response.message.content)
# Ask the local Ollama service for its installed model list.
models = ollama.list()

print("Ollama service is reachable.")
print("Installed local models:")
# Use one model consistently throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"
# Display the local models available to this notebook.
for model in models.models:
    print(f"- {model.model}")

SYSTEM = (
    "You are a helpful assistant. Answer using ONLY the provided context. "
    "If the context does not contain the answer, say so plainly. Cite the "
    "ticket id in square brackets after any fact you use."
)

def ask_qdrant_rag(question: str, collection: str, k: int = 3) -> dict:
    """Full mini-RAG: embed → Qdrant retrieve → prompt → generate."""
    # 1. Embed the query
    q_vec = embed_one_minilm(question)
    
    # 2. Retrieve top-K from Qdrant
    hits = qdrant.query_points(
        collection_name=collection,
        query=q_vec,
        limit=k,
    ).points
    
    # 3. Build the prompt with retrieved chunks
    context = "\n\n".join(
        f"[{h.payload['ticket_id']}]\n{h.payload['text']}"
        for h in hits
    )
    
    # 4. Generate
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content":
                f"Context:\n{context}\n\n---\n\nQuestion: {question}"},
        ],
        # FIX 1: Pass temperature inside the options dictionary
        options={"temperature": 0.0}
    )
    
    return {
        "question": question,
        # FIX 2: Parse the Ollama response dictionary correctly
        "answer":   resp['message']['content'],
        "sources":  [h.payload["ticket_id"] for h in hits],
    }

# Test on one question
result = ask_qdrant_rag("Why are users failing to authenticate to the dashboard?", COLL_COSINE)
print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources: {result['sources']}")

Note: you may need to restart the kernel to use updated packages.
Recall
Ollama service is reachable.
Installed local models:
- gpt-oss:20b
Q: Why are users failing to authenticate to the dashboard?

A: Users are failing to authenticate to the dashboard because the Okta SSO integration is not working – the login flow is returning an HTTP 401 Unauthorized error [tkt_auth_01]. Additionally, the Active Directory group permissions for new hires have been dropped, so the necessary IAM access tokens for AWS production are missing, which can also prevent successful authentication [tkt_auth_02].

Sources: ['tkt_auth_01', 'tkt_auth_02', 'tkt_net_02']


---

## Cell 15 — Run all 5 test questions

Now the full test set. Watch how well the mini-RAG handles each.

In [27]:
# Define the test questions for IT incident tickets
TEST_QUESTIONS = [
    {
        "q": "Why are users failing to authenticate to the dashboard?",
        "expected_categories": ["auth"]
    },
    {
        "q": "What is causing high latency in our databases?",
        "expected_categories": ["database"]
    },
    {
        "q": "Are there any issues with Jenkins pipelines?",
        "expected_categories": ["cicd"]
    },
    {
        "q": "Why did we get a cloud cost alert?",
        "expected_categories": ["billing"]
    }
]

for tq in TEST_QUESTIONS:
    result = ask_qdrant_rag(tq["q"], COLL_COSINE, k=3)
    print(f"── {tq['q']}")
    print(f"   expected categories: {tq['expected_categories']}")
    print(f"   sources retrieved:   {result['sources']}")
    print(f"   answer: {result['answer']}")
    print()

── Why are users failing to authenticate to the dashboard?
   expected categories: ['auth']
   sources retrieved:   ['tkt_auth_01', 'tkt_auth_02', 'tkt_net_02']
   answer: Users are failing to authenticate to the dashboard because the Okta SSO integration is not working – the login flow is returning an HTTP 401 Unauthorized error [tkt_auth_01]. Additionally, the Active Directory group permissions for new hires have been dropped, so the necessary IAM access tokens for AWS production are missing, which can also prevent successful authentication [tkt_auth_02].

── What is causing high latency in our databases?
   expected categories: ['database']
   sources retrieved:   ['tkt_db_02', 'tkt_db_01', 'tkt_bill_01']
   answer: High latency in the databases is being caused by two separate issues:

1. **Redis cache cluster** – its high memory utilization is leading to latency spikes in query resolution. [tkt_db_02]  
2. **PostgreSQL primary instance** – connection pool exhaustion during peak rea

**Discussion:**
- Did every question retrieve at least one ticket from the expected category?
- Which question was hardest?
- Did the LLM correctly cite `[ticket_id]` for each fact it used?
- Look at Question 4 ("Why did we get a cloud cost alert?") — did AWS and SaaS
  come back? Did a pipeline sneak in?

---

## Cell 16 — Cleanup

Delete the collections we created today so your Qdrant cluster stays tidy.

(In your capstone Track B work you'll create ONE persistent collection —
`capstone_chunks`. These demo collections are transient.)

In [28]:
for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED, COLL_QUANT,
             COLL_COSINE, COLL_DOT, COLL_L2]:
    try:
        qdrant.delete_collection(coll)
        print(f"  deleted {coll}")
    except Exception as e:
        print(f"  skip {coll}: {e}")

print("\nDay 1's tickets collection (wk07_day1_tickets) is preserved.")
print("Delete manually if you don't want it.")

  deleted wk07_day2_hnsw_default
  deleted wk07_day2_hnsw_tuned
  deleted wk07_day2_quantized
  deleted wk07_day2_metric_cosine
  deleted wk07_day2_metric_dot
  deleted wk07_day2_metric_l2

Day 1's tickets collection (wk07_day1_tickets) is preserved.
Delete manually if you don't want it.


---

## Cell 17 — Wrap: what you learned across two days

**Day 1 — Embedding space & Local Models:**

1. Swapped cloud APIs for local models. Used **MiniLM** for 384-dimensional embeddings and **Ollama (`gpt-oss:20b`)** for generation. Zero API costs and full data privacy.
2. Built intuition for semantic geometry — categories cluster.
3. Toured 4 vector DBs. Programme default: Qdrant.
4. Made your first Qdrant call.

**Day 2 — Qdrant tour:**
5. **Indices** exist because linear scan doesn't scale beyond ~10K docs.
6. **HNSW** is Qdrant's default. Two knobs: `m` (connectivity) and
`ef_construct` (build depth). Both trade recall vs cost.
7. **Quantization** compresses vectors (int8 = 4× smaller). Pays off at scale.
8. **Cosine, dot, L2** — three distance metrics. Cosine is the safe default.
9. **The silent-bug pattern** — non-normalised vectors + dot product = wrong
rankings that look fine. Stay on cosine.
10. **Local Mini-RAG on Qdrant** — same shape as W6's ask_rag, but powered entirely by local models and local vector storage.

**Track B (take-home): migrate YOUR capstone.**

Open `AI-RAG_W7_Application_Growth_Guide.md` and follow the steps. Same
pattern you just did with 10 tickets, applied to your capstone's 100-300
real chunks. About 2 hours self-paced.

You will:
- Add `src/rag/qdrant_store.py` (~60 lines)
- Add `src/rag/qdrant_rag.py` (~80 lines) — parallel to W6's naive_rag
- Add `scripts/migrate_to_qdrant.py` — one-off migration
- Swap one function call in `src/api/main.py`
- Update `docs/adr/0001-capstone-framing.md` with vector-stack decisions
- Fill `docs/kpi/wk7-snapshot.md`